# 02 Malaysia Spatial Features

Validate Malaysia state-level satellite/geospatial features and regenerate report EDA figures.

This notebook uses the final state-year panel. It regenerates analytical report figures from active state-level data only.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

QA_DIR = PROJECT_ROOT / "outputs" / "qa"
REPORTING_DIR = PROJECT_ROOT / "outputs" / "reporting"
INTERMEDIATE_DIR = PROJECT_ROOT / "outputs" / "intermediate"
for path in [QA_DIR, REPORTING_DIR, INTERMEDIATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict) -> dict:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    return payload

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

In [2]:
import matplotlib.pyplot as plt
import numpy as np

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(PROJECT_ROOT / "data" / "state_year_panel_modelready_2002_2022.csv")
panel = panel[panel["year"].isin([2002, 2004, 2007, 2009, 2012, 2014, 2016, 2019, 2020, 2022])].copy()

proxy_cols = [
    "wmean_ntl_mean", "wmean_evi_mean", "wmean_ndvi_mean",
    "wmean_frac_urban", "pop_sum_grid", "wmean_elevation", "wmean_slope",
]
target_cols = ["poverty_absolute", "poverty_relative", "poverty_hardcore"]
existing_proxy_cols = [col for col in proxy_cols if col in panel.columns]
existing_target_cols = [col for col in target_cols if col in panel.columns]

# Figure 3: NTL vs absolute poverty scatter.
fig, ax = plt.subplots(figsize=(7.2, 4.6))
sc = ax.scatter(
    panel["wmean_ntl_mean"], panel["poverty_absolute"],
    c=panel["year"], cmap="viridis", s=42, alpha=0.82,
    edgecolor="white", linewidth=0.4,
)
ax.set_title("Nighttime Light Intensity vs Absolute Poverty")
ax.set_xlabel("Population-weighted mean NTL")
ax.set_ylabel("Absolute poverty (%)")
ax.grid(alpha=0.2)
cb = fig.colorbar(sc, ax=ax)
cb.set_label("Survey year")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig4_4_ntl_poverty_scatter.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Figure 4: poverty target distributions.
fig, axes = plt.subplots(1, len(existing_target_cols), figsize=(10, 3.4), sharey=False)
if len(existing_target_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, existing_target_cols):
    vals = panel[col].dropna()
    ax.hist(vals, bins=14, color="#2E75B6", edgecolor="white", alpha=0.9)
    ax.axvline(vals.median(), color="#C00000", linewidth=1.6, label="Median")
    ax.set_title(col.replace("poverty_", "").title())
    ax.set_xlabel("Poverty rate (%)")
    ax.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("State-years")
axes[-1].legend(frameon=False, fontsize=8)
fig.suptitle("Distribution of State-Level Poverty Rates Across Survey Years", y=1.03)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig4_3_poverty_distributions.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Figure 5: temporal stability of proxy-poverty correlations.
corr_rows = []
for year, group in panel.groupby("year"):
    for proxy in ["wmean_ntl_mean", "wmean_evi_mean", "wmean_ndvi_mean", "wmean_frac_urban"]:
        if proxy in group.columns:
            sub = group[[proxy, "poverty_absolute"]].dropna()
            corr = sub[proxy].corr(sub["poverty_absolute"], method="spearman") if len(sub) > 3 else np.nan
            corr_rows.append({"year": int(year), "proxy": proxy, "spearman": corr})
corr_df = pd.DataFrame(corr_rows)
fig, ax = plt.subplots(figsize=(8, 4.4))
for proxy, group in corr_df.groupby("proxy"):
    ax.plot(group["year"], group["spearman"], marker="o", linewidth=1.7, label=proxy.replace("wmean_", ""))
ax.axhline(0, color="#555", linewidth=0.8)
ax.axvline(2011, color="#777", linestyle="--", linewidth=1.0)
ax.text(2011.2, 0.92, "DMSP/VIIRS", fontsize=8, color="#555")
ax.set_title("Temporal Stability of Proxy-Poverty Correlations")
ax.set_xlabel("Survey year")
ax.set_ylabel("Spearman r with absolute poverty")
ax.set_ylim(-1, 1)
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig4_1_temporal_stability.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Figure 6: feature/target correlation heatmap.
heat_cols = existing_target_cols + existing_proxy_cols[:8]
corr = panel[heat_cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(8.2, 6.4))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
labels = [c.replace("poverty_", "pov_").replace("wmean_", "") for c in corr.columns]
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Spearman r")
ax.set_title("Feature Correlation Matrix - State-Year Panel")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig4_2_correlation_heatmap.png", dpi=180, bbox_inches="tight")
plt.close(fig)

summary = {
    "active_scope": "state_level_only",
    "feature_rows": int(len(panel)),
    "feature_columns": int(len(panel.columns)),
    "years": sorted(int(y) for y in panel["year"].dropna().unique()),
    "states": int(panel["state_key"].nunique()) if "state_key" in panel else int(panel["state"].nunique()),
    "proxy_columns_checked": existing_proxy_cols,
    "target_columns_checked": existing_target_cols,
    "generated_figures": [
        "outputs/figures/fig4_4_ntl_poverty_scatter.png",
        "outputs/figures/fig4_3_poverty_distributions.png",
        "outputs/figures/fig4_1_temporal_stability.png",
        "outputs/figures/fig4_2_correlation_heatmap.png",
    ],
}
write_json(QA_DIR / "malaysia_spatial_feature_summary.json", summary)
pd.DataFrame([summary])

,active_scope,feature_rows,feature_columns,years,states,proxy_columns_checked,target_columns_checked,generated_figures
0,state_level_only,158,70,"[2002, 2004, 2007, 2009, 2012, 2014, 2016, 201...",16,"[wmean_ntl_mean, wmean_evi_mean, wmean_ndvi_me...","[poverty_absolute, poverty_relative, poverty_h...",[outputs/figures/fig4_4_ntl_poverty_scatter.pn...
